# SRO Site Telemetry Sandbox

This notebook is a sandbox for learning how to read and interpret the SRO site telemetry
files for builidng roof status and weather.

## Mounting SRO Files

The SRO server is 192.168.1.57, we map these drives
<pre>
    SRO14-roof -> R:
    SRO-Weather -> W:
</pre>
onto the DEMONEXT PC file system.

### Important Testing Note

This notebook is just for raw testing and is not integrated with the `demonext` module. See `site.py` in the `demonext` module, and the `Site_Sandbox.ipynb` notebook for a version that uses the beta release of the DEMONEXT flight code.

In [307]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, coordinates, and times

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import TETE
from astropy.time import Time

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

# Import all demonext modules

import demonext
from demonext import config, telescope, pdu, camera, focuser, obsfile

# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}


## Data-taking session startup

**This assumes you have already started the observatory subsystems using the `StartUp_Shutdown.ipynb` notebook**


In [310]:
# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# Read the runtime configuration file

try:
    cfg = config.Config(cfgFile)
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")


### Connect to DEMONEXT observatory systems

Connect to the follwing services, in order:
 * Site services: weather and roof status


In [313]:

weatherDir = demonext.homePath(cfg.config["directories"]["WeatherDir"])
weatherFile = Path(weatherDir) / cfg.config["telemetry"]["WxFile"]

roofDir = demonext.homePath(cfg.config["directories"]["RoofDir"])
roofFile = Path(roofDir) / cfg.config["telemetry"]["RoofFile"]

if roofFile.exists():
    print(f"Building Roof Data: {roofFile}")
else:
    print(f"ERROR: Building roof file {roofFile} not found")
    
if weatherFile.exists():
    print(f"SRO Weather Data: {weatherFile}")
else:
    print(f"ERROR: SRO weather data file {weatherFile} not found")
    

Building Roof Data: R:\RoofStatusFile.txt
SRO Weather Data: W:\sroweather.txt


## Read and parse the roof data

Open and read in the roof data. We build a `roof` dictionary with the info we need.

In [316]:
def getRoof(roofFile):
    roofStatus = {}
    try:
        with open(roofFile,"r",encoding="utf-8") as file:
            data = file.read()
            roofData = data.strip().split()
            roofStatus["date"] = roofData[0][3:] # first 3 bytes are ???, skip
            roofStatus["time"] = roofData[1][:8]
            roofStatus["iso"] = f"{roofData[0][3:]}T{roofData[1][:8]}"
            roofStatus["position"] = roofData[4]
            if roofData[4] == "OPEN":
                roofStatus["open"] = True
            else:
                roofStatus["open"] = False
    except FileNotFoundError:
        print(f"Error: Roof status file {roofFile} not found")
        roofStatus["open"] = False
        roofStatus["position"] = "UNKNOWN"
    except IOError as err:
        print(f"Error reading roof file {roofFile} - {err}")
        roofStatus["open"] = False
        roofStatus["position"] = "UNKNOWN"
    return roofStatus

roof = getRoof(roofFile)
print("roof Dict: ",roof)
if roof['open']:
    print(f"Roof is Open")
else:
    print(f"Roof is {roof['position']}")


roof Dict:  {'date': '2026-03-19', 'time': '01:59:15', 'iso': '2026-03-19T01:59:15', 'position': 'CLOSED', 'open': False}
Roof is CLOSED


## Read and parse weather data

Weather data is stored in "Boltwood II" one-line format (https://interactiveastronomy.com/skyroof_help/Weatherdatafile.html).

We build a `weather` dictionary with the info we need.

### Conversions

Some helpful conversion functions:
 * degC = (degF - 32)/1.8
 * m/sec = (12 in/ft * 5280 ft/mi * 0.0254 m/in) * mph / 3600 sec/h
 * m/sec = 1858 m/nm * knots / 3600 sec/h

In [319]:
# convert F to C

def f2c(tempF):
    return (tempF - 32.0)/1.8

# convert mph to m/s

def mph2ms(mph):
    return (12*5280*0.0254)*mph/3600.

# convert knots to m/s. 1 nautical mile = 1852 meters exactly

def knots2ms(knots):
    return 1852.0*knots/3600.0

# retrieve weather data, read and parse into a dictionary

def getWeather(wxFile):
    weather = {}

    # Translations of different flag codes
    
    cloudFlags = ["Unknown","clear","light clouds","cloudy"]
    windFlags = ["Unknown","calm","windy","very windy"]
    rainFlags = ["Unknown","dry","damp","rain"]
    darkFlags = ["Unknown","dark","dim","daylight"]

    # Read and parse the weather file
    
    try:
        with open(wxFile,"r",encoding="utf-8") as file:
            data = file.read()
            wxData = data.strip().split()
            # print(wxData)
            weather["up"] = True
            weather["date"] = wxData[0]
            weather["time"] = wxData[1]
            weather["iso"] = f"{wxData[0]}T{wxData[1][:8]}"
            tempUnits = wxData[2]
            windUnits = wxData[3]

            # convert temperatures from fahrenheit to celsius as needed
            
            if tempUnits == "F":
                weather["skyTemp"] = f2c(float(wxData[4]))
                weather["airTemp"] =  f2c(float(wxData[5]))
                weather["dewpoint"] = f2c(float(wxData[9]))
            else:
                weather["skyTemp"] = float(wxData[4])
                weather["airTemp"] = float(wxData[5])
                weather["dewpoint"] = float(wxData[9])

            # convert wind speed from mph or knots to m/s as needed
            
            if windUnits == "M":
                weather["windSpeed"] = mph2ms(float(wxData[7]))
            else:
                weather["windSpeed"] = knots2ms(float(wxData[7]))
            
            weather["humidity"] = float(wxData[8])
            weather["rain"] = rainFlags[int(wxData[17])]
            weather["damp"] = int(wxData[12])
            weather["clouds"] = cloudFlags[int(wxData[15])]
            weather["wind"] = windFlags[int(wxData[16])]
            weather["dark"] = darkFlags[int(wxData[18])] 
    except FileNotFoundError:
        print(f"Error: SRO weather file {wxFile} not found")
        weather["up"] = False
    except IOError as err:
        print(f"Error SRO weather file {wxFile} - {err}")
        weather["up"] = False
    return weather

weather = getWeather(weatherFile)
print(weather)

{'up': True, 'date': '2026-03-19', 'time': '13:59:21.00', 'iso': '2026-03-19T13:59:21', 'skyTemp': -12.722222222222221, 'airTemp': 24.944444444444446, 'dewpoint': 7.833333333333334, 'windSpeed': 0.44703999999999994, 'humidity': 34.0, 'rain': 'dry', 'damp': 0, 'clouds': 'clear', 'wind': 'calm', 'dark': 'daylight'}


## Quick Look at both

Formatted output

In [322]:
roof = getRoof(roofFile)
if roof['open']:
    print(f"Building 14 roof is OPEN ({roof['iso']})")
else:
    print(f"Building 14 roof is {roof['position']} ({roof['iso']})")
    
weather = getWeather(weatherFile)
if weather['up']:
    print(f"\nSRO Weather ({weather['date']} {weather['time']} PT):")
    print(f"      Temp: {weather['airTemp']:.1f} C")
    print(f"  Humidity: {weather['humidity']:.1f}%  dewPoint: {weather['dewpoint']:.1f} C")
    print(f"      Wind: {weather['wind']}, {weather['windSpeed']:.1f} m/s")
    print(f"       Sky: {weather['clouds']}, {weather['rain']}, {weather['dark']}")
else:
    print(f"\nWeather station not readable")

Building 14 roof is CLOSED (2026-03-19T01:59:25)

SRO Weather (2026-03-19 13:59:21.00 PT):
      Temp: 24.9 C
  Humidity: 34.0%  dewPoint: 7.8 C
      Wind: calm, 0.4 m/s
       Sky: clear, dry, daylight


## Tests

### How often is weather updated?

**Answer:** Every 20 seconds (180 times per hour)


In [325]:
runTests = False

numQueries = 20
cadence = 10

if runTests:
    for i in range(numQueries):
        weather = getWeather(weatherFile)
        print(f"  {weather['time']} T={weather['airTemp']:.1f} C")
        time.sleep(cadence)
    print("done")

### how often is roof status updated?

**Answer:** Every 10 seconds (360 times per hour)

In [328]:
numQueries = 20
cadence = 10

if runTests:
    for i in range(numQueries):
        roof = getRoof(roofFile)
        print(f"  {roof['time']} Roof is {roof['position']}")
        time.sleep(cadence)
    print("done")